In [1]:
import os
import cv2
!mkdir Data2
events_list = os.listdir("/kaggle/input/football-data/Data")
for event in events_list:
  os.mkdir("./Data2/" + event)
  for files in os.listdir("/kaggle/input/football-data/Data/" +event):
    os.mkdir("./Data2/" + event + "/" + files.replace(".mp4",""))
    vidcap = cv2.VideoCapture("/kaggle/input/football-data/Data/" + event + "/" + files)
    success,image = vidcap.read()
    count = 0
    while success:
      cv2.imwrite("./Data2/" + event + "/" + files.replace(".mp4","") +  "/img_" + "%04d.jpg" % (count,), image)     # save frame as JPEG file      
      success,image = vidcap.read()
      count += 1

In [2]:
import os
var = -1
lst = []
for events in os.listdir("/kaggle/working/Data2"):
    for files in os.listdir("/kaggle/working/Data2/" + events):
        if events == "Goal":
            var = 0
        elif events == "Miss":
            var = 1
        elif events == "Cards":
            var = 2
        num_frames = len(os.listdir("/kaggle/working/Data2/" + events + "/" + files))
        f = open("/kaggle/working/Data2/annotations.txt", "a")
        f.write(events + "/" + files + " " + str(1) + " " + str(num_frames) + " " + str(var) + "\n")
        f.close()

In [3]:
import os
import os.path
import numpy as np
from PIL import Image
from torchvision import transforms
import torch
from typing import List, Union, Tuple, Any
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import ImageGrid
import torch.optim as optim

In [4]:
class VideoRecord(object):
    """
    Helper class for class VideoFrameDataset. This class
    represents a video sample's metadata.
    Args:
        root_datapath: the system path to the root folder
                       of the videos.
        row: A list with four or more elements where 1) The first
             element is the path to the video sample's frames excluding
             the root_datapath prefix 2) The  second element is the starting frame id of the video
             3) The third element is the inclusive ending frame id of the video
             4) The fourth element is the label index.
             5) any following elements are labels in the case of multi-label classification
    """
    def __init__(self, row, root_datapath):
        self._data = row
        self._path = os.path.join(root_datapath, row[0])


    @property
    def path(self) -> str:
        return self._path

    @property
    def num_frames(self) -> int:
        return self.end_frame - self.start_frame + 1  # +1 because end frame is inclusive
    @property
    def start_frame(self) -> int:
        return int(self._data[1])

    @property
    def end_frame(self) -> int:
        return int(self._data[2])

    @property
    def label(self) -> Union[int, List[int]]:
        # just one label_id
        if len(self._data) == 4:
            return int(self._data[3])
        # sample associated with multiple labels
        else:
            return [int(label_id) for label_id in self._data[3:]]


In [5]:
class VideoFrameDataset(torch.utils.data.Dataset):
    r"""
    A highly efficient and adaptable dataset class for videos.
    Instead of loading every frame of a video,
    loads x RGB frames of a video (sparse temporal sampling) and evenly
    chooses those frames from start to end of the video, returning
    a list of x PIL images or ``FRAMES x CHANNELS x HEIGHT x WIDTH``
    tensors where FRAMES=x if the ``ImglistToTensor()``
    transform is used.
    More specifically, the frame range [START_FRAME, END_FRAME] is divided into NUM_SEGMENTS
    segments and FRAMES_PER_SEGMENT consecutive frames are taken from each segment.
    Note:
        A demonstration of using this class can be seen
        in ``demo.py``
        https://github.com/RaivoKoot/Video-Dataset-Loading-Pytorch
    Note:
        This dataset broadly corresponds to the frame sampling technique
        introduced in ``Temporal Segment Networks`` at ECCV2016
        https://arxiv.org/abs/1608.00859.
    Note:
        This class relies on receiving video data in a structure where
        inside a ``ROOT_DATA`` folder, each video lies in its own folder,
        where each video folder contains the frames of the video as
        individual files with a naming convention such as
        img_001.jpg ... img_059.jpg.
        For enumeration and annotations, this class expects to receive
        the path to a .txt file where each video sample has a row with four
        (or more in the case of multi-label, see README on Github)
        space separated values:
        ``VIDEO_FOLDER_PATH     START_FRAME      END_FRAME      LABEL_INDEX``.
        ``VIDEO_FOLDER_PATH`` is expected to be the path of a video folder
        excluding the ``ROOT_DATA`` prefix. For example, ``ROOT_DATA`` might
        be ``home\data\datasetxyz\videos\``, inside of which a ``VIDEO_FOLDER_PATH``
        might be ``jumping\0052\`` or ``sample1\`` or ``00053\``.
    Args:
        root_path: The root path in which video folders lie.
                   this is ROOT_DATA from the description above.
        annotationfile_path: The .txt annotation file containing
                             one row per video sample as described above.
        num_segments: The number of segments the video should
                      be divided into to sample frames from.
        frames_per_segment: The number of frames that should
                            be loaded per segment. For each segment's
                            frame-range, a random start index or the
                            center is chosen, from which frames_per_segment
                            consecutive frames are loaded.
        imagefile_template: The image filename template that video frame files
                            have inside of their video folders as described above.
        transform: Transform pipeline that receives a list of PIL images/frames.
        test_mode: If True, frames are taken from the center of each
                   segment, instead of a random location in each segment.
    """
    def __init__(self,
                 root_path: str,
                 annotationfile_path: str,
                 num_segments: int = 3,
                 frames_per_segment: int = 1,
                 imagefile_template: str='img_{:04d}.jpg',
                 transform = None,
                 test_mode: bool = False):
        super(VideoFrameDataset, self).__init__()

        self.root_path = root_path
        self.annotationfile_path = annotationfile_path
        self.num_segments = num_segments
        self.frames_per_segment = frames_per_segment
        self.imagefile_template = imagefile_template
        self.transform = transform
        self.test_mode = test_mode

        self._parse_annotationfile()
        self._sanity_check_samples()

    def _load_image(self, directory: str, idx: int) -> Image.Image:
        return Image.open(os.path.join(directory, self.imagefile_template.format(idx))).convert('RGB')

    def _parse_annotationfile(self):
        self.video_list = [VideoRecord(x.strip().split(), self.root_path) for x in open(self.annotationfile_path)]

    def _sanity_check_samples(self):
        for record in self.video_list:
            if record.num_frames <= 0 or record.start_frame == record.end_frame:
                print(f"\nDataset Warning: video {record.path} seems to have zero RGB frames on disk!\n")

            elif record.num_frames < (self.num_segments * self.frames_per_segment):
                print(f"\nDataset Warning: video {record.path} has {record.num_frames} frames "
                      f"but the dataloader is set up to load "
                      f"(num_segments={self.num_segments})*(frames_per_segment={self.frames_per_segment})"
                      f"={self.num_segments * self.frames_per_segment} frames. Dataloader will throw an "
                      f"error when trying to load this video.\n")

    def _get_start_indices(self, record: VideoRecord) -> 'np.ndarray[int]':
        """
        For each segment, choose a start index from where frames
        are to be loaded from.
        Args:
            record: VideoRecord denoting a video sample.
        Returns:
            List of indices of where the frames of each
            segment are to be loaded from.
        """
        # choose start indices that are perfectly evenly spread across the video frames.
        if self.test_mode:
            distance_between_indices = (record.num_frames - self.frames_per_segment + 1) / float(self.num_segments)

            start_indices = np.array([int(distance_between_indices / 2.0 + distance_between_indices * x)
                                      for x in range(self.num_segments)])
        # randomly sample start indices that are approximately evenly spread across the video frames.
        else:
            max_valid_start_index = (record.num_frames - self.frames_per_segment + 1) // self.num_segments

            start_indices = np.multiply(list(range(self.num_segments)), max_valid_start_index) + \
                      np.random.randint(max_valid_start_index, size=self.num_segments)

        return start_indices

    def __getitem__(self, idx: int) -> Union[
        Tuple[List[Image.Image], Union[int, List[int]]],
        Tuple['torch.Tensor[num_frames, channels, height, width]', Union[int, List[int]]],
        Tuple[Any, Union[int, List[int]]],
        ]:
        """
        For video with id idx, loads self.NUM_SEGMENTS * self.FRAMES_PER_SEGMENT
        frames from evenly chosen locations across the video.
        Args:
            idx: Video sample index.
        Returns:
            A tuple of (video, label). Label is either a single
            integer or a list of integers in the case of multiple labels.
            Video is either 1) a list of PIL images if no transform is used
            2) a batch of shape (NUM_IMAGES x CHANNELS x HEIGHT x WIDTH) in the range [0,1]
            if the transform "ImglistToTensor" is used
            3) or anything else if a custom transform is used.
        """
        record: VideoRecord = self.video_list[idx]

        frame_start_indices: 'np.ndarray[int]' = self._get_start_indices(record)

        return self._get(record, frame_start_indices)

    def _get(self, record: VideoRecord, frame_start_indices: 'np.ndarray[int]') -> Union[
        Tuple[List[Image.Image], Union[int, List[int]]],
        Tuple['torch.Tensor[num_frames, channels, height, width]', Union[int, List[int]]],
        Tuple[Any, Union[int, List[int]]],
        ]:
        """
        Loads the frames of a video at the corresponding
        indices.
        Args:
            record: VideoRecord denoting a video sample.
            frame_start_indices: Indices from which to load consecutive frames from.
        Returns:
            A tuple of (video, label). Label is either a single
            integer or a list of integers in the case of multiple labels.
            Video is either 1) a list of PIL images if no transform is used
            2) a batch of shape (NUM_IMAGES x CHANNELS x HEIGHT x WIDTH) in the range [0,1]
            if the transform "ImglistToTensor" is used
            3) or anything else if a custom transform is used.
        """

        frame_start_indices = frame_start_indices + record.start_frame
        images = list()




        # from each start_index, load self.frames_per_segment
        # consecutive frames
        for start_index in frame_start_indices:
            frame_index = int(start_index)

            # load self.frames_per_segment consecutive frames
            for _ in range(self.frames_per_segment):
                try:
                    image = self._load_image(record.path, frame_index)
                    images.append(image)
                except:
                    continue

                if frame_index < record.end_frame:
                    frame_index += 1

        if self.transform is not None:
            images = self.transform(images)


        #  MY MODIFICATION 

        if images.shape[0] != self.num_segments*self.frames_per_segment:
            remainingframes = (self.num_segments*self.frames_per_segment) - images.shape[0]
            black_image = torch.zeros((remainingframes,images.shape[1],images.shape[2],images.shape[3]))
            try:
                images = torch.stack(images,black_image)
            except:
                images = torch.cat((images,black_image),dim=0)


        images = torch.fft.fft2(images)
        images = torch.fft.fftshift(images)
        images= 20*np.log(np.abs(images))

        return images, record.label

    def __len__(self):
        return len(self.video_list)


In [6]:
class ImglistToTensor(torch.nn.Module):
    """
    Converts a list of PIL images in the range [0,255] to a torch.FloatTensor
    of shape (NUM_IMAGES x CHANNELS x HEIGHT x WIDTH) in the range [0,1].
    Can be used as first transform for ``VideoFrameDataset``.
    """
    @staticmethod
    def forward(img_list: List[Image.Image]) -> 'torch.Tensor[NUM_IMAGES, CHANNELS, HEIGHT, WIDTH]':
        """
        Converts each PIL image in a list to
        a torch Tensor and stacks them into
        a single tensor.
        Args:
            img_list: list of PIL images.
        Returns:
            tensor of size ``NUM_IMAGES x CHANNELS x HEIGHT x WIDTH``
        """
        return torch.stack([transforms.functional.to_tensor(pic) for pic in img_list])

In [7]:
def plot_video(rows, cols, frame_list, plot_width, plot_height, title: str):
    fig = plt.figure(figsize=(plot_width, plot_height))
    grid = ImageGrid(fig, 111,  # similar to subplot(111)
                     nrows_ncols=(rows, cols),  # creates 2x2 grid of axes
                     axes_pad=0.3,  # pad between axes in inch.
                     )

    for index, (ax, im) in enumerate(zip(grid, frame_list)):
        # Iterating over the grid returns the Axes.
        ax.imshow(im)
        ax.set_title(index)
    plt.suptitle(title)
    plt.show()

In [8]:
videos_root = os.path.join(os.getcwd(), '/kaggle/working/Data2')
annotation_file = os.path.join(videos_root, '/kaggle/working/Data2/annotations.txt')

In [9]:
preprocess = transforms.Compose([
    ImglistToTensor(),  # list of PIL images to (FRAMES x CHANNELS x HEIGHT x WIDTH) tensor
    transforms.Resize(224),  # image batch, resize smaller edge to 299
    # transforms.CenterCrop(299),  # image batch, center crop to square 299x299
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.Grayscale(1)
])


dataset = VideoFrameDataset(
    root_path=videos_root,
    annotationfile_path=annotation_file,
    num_segments=37,
    frames_per_segment=2,
    imagefile_template='img_{:04d}.jpg',
    transform=preprocess,
    test_mode=True
)

In [10]:
device = torch.device('cuda:0') 
sample = dataset[90]
frame_tensor = sample[0]
label = sample[1]

In [11]:
trainloader = torch.utils.data.DataLoader(dataset, batch_size=2, shuffle=True)

In [12]:
import torch.nn as nn
import torch.nn.functional as F


class VGG16(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Sequential(nn.Conv3d(1,64,(3,3,1)))
        self.conv2 = nn.Sequential(nn.Conv3d(64,64,(3,3,1)),nn.MaxPool3d(2,(1,2,2)))

        self.conv3 = nn.Sequential(nn.Conv3d(64,128,3),nn.ReLU())
        self.conv4 = nn.Sequential(nn.Conv3d(128,128,3),nn.ReLU(),nn.MaxPool3d(2,(1,2,2)))

        self.conv5 = nn.Sequential(nn.Conv3d(128,256,3),nn.ReLU())
        self.conv6 = nn.Sequential(nn.Conv3d(256,256,3),nn.ReLU())
        self.conv7 = nn.Sequential(nn.Conv3d(256,256,3),nn.ReLU(),nn.MaxPool3d(2,2))

        self.conv8 = nn.Sequential(nn.Conv3d(256,512,3),nn.ReLU())
        self.conv9 = nn.Sequential(nn.Conv3d(512,512,3),nn.ReLU())
        self.conv10 = nn.Sequential(nn.Conv3d(512,512,3),nn.ReLU(),nn.MaxPool3d(2,2))

        self.conv11 = nn.Sequential(nn.Conv3d(512,512,3),nn.ReLU())
        self.conv12 = nn.Sequential(nn.Conv3d(512,512,3),nn.ReLU())
        self.conv13 = nn.Sequential(nn.Conv3d(512,512,3),nn.ReLU(),nn.MaxPool3d(2,2))

        self.FC1 = nn.Sequential(nn.Linear(6144,4096),nn.ReLU())
        self.FC2 = nn.Sequential(nn.Linear(4096,4096),nn.ReLU())
        self.FC3 = nn.Sequential(nn.Linear(4096,1000),nn.ReLU())
        self.FC4 = nn.Sequential(nn.Linear(1000,3),nn.ReLU())


    def forward(self,x):
        
        
        x = self.conv1(x)
        x = self.conv2(x)            
                      
            
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.conv5(x)
        x = self.conv6(x)

        
        x = self.conv7(x)
        x = self.conv8(x)
        

        x = self.conv9(x)
        x = self.conv10(x)
  
        x = self.conv11(x)
        x = self.conv12(x)
        x = self.conv13(x)

        x = torch.flatten(x, 1) 
        x = self.FC1(x)
        x = self.FC2(x)
        x = self.FC3(x)
        x = self.FC4(x)
        print(x.shape)
        
        return x




In [13]:
Network = VGG16()
# Network = Network.cuda()

In [14]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(Network.parameters(), lr=0.001, momentum=0.9)
def accuracy(out, labels):
    _,pred = torch.max(out, dim=1)
    return torch.sum(pred==labels).item()

In [15]:
for epoch in range(10):
    running_loss = 0.0
    for video_batch, labels in trainloader:

        inputs = video_batch
#         inputs, labels = video_batch.cuda(), labels.cuda()
        inputs = torch.moveaxis(inputs,1,2)
        optimizer.zero_grad()
        outputs = Network(inputs)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    if (epoch+1) % 2 == 0:
          print ('Epoch [{}/{}], Step [{}/{}], Loss: {:.4f}' 
                  .format(epoch+1, 10, i+1, total_step, loss.item()))

torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([

NameError: name 'i' is not defined

In [ ]:
correct = 0
total = 0
# since we're not training, we don't need to calculate the gradients for our outputs
with torch.no_grad():
    for images,labels in trainloader:
#         images, labels = data
        # calculate outputs by running images through the network
#         images, labels = images.cuda(), labels.cuda()
        images = torch.moveaxis(images,1,2)
        outputs = Network(images)
        # the class with the highest energy is what we choose as prediction
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy of the network on the 10000 test images: {100 * correct // total} %')